# Part 5: Epistemic Flattening Detector

**What this is:** The first automated tool that detects when an LLM intermediary has flattened the epistemic stance of a source utterance. Nobody has built this before.

**What it does:** Takes a (source sentence, reformulation) pair and outputs a flattening score between 0 and 1. If the score is above 0.5, the reformulation has likely lost the speaker's uncertainty.

**How it works:**
1. Extracts 20 features from each (source, reformulation) pair: hedge word counts, preservation rates, booster additions, length changes
2. Trains 3 classifiers (Logistic Regression, Decision Tree, Random Forest) on the 539 reformulations from Part 2
3. Evaluates on held-out test set with precision, recall, F1, and AUC-ROC

**Input:** `experiment_results.json` from Part 2

**Output:** Trained detector, evaluation metrics, feature importance analysis

In [ ]:
from google.colab import files
import json, re, csv, os
import numpy as np
from collections import Counter, defaultdict

print('Upload experiment_results.json from Part 2:')
uploaded = files.upload()

In [ ]:
with open('experiment_results.json') as f:
    data = json.load(f)
results = data['results']
valid = [r for r in results if r.get('parsed_json') and not r.get('parse_error')]
dim1 = [r for r in valid if r['condition'] in ('1A', '1B', '1C')]
print(f'Loaded {len(dim1)} Dimension 1 reformulations')

## Step 1: Feature Extraction

For each (source, reformulation) pair, we extract 20 features:
- **Hedge counts** in source and reformulation (strong/moderate/weak)
- **Hedge preservation rate** (what fraction of source hedges survived)
- **Hedge words dropped/added** (specific words that disappeared or appeared)
- **Booster counts** (words like 'clearly', 'demonstrates', 'proves' that signal over-confidence)
- **Lexical overlap** (Jaccard similarity between source and reformulation)
- **Length ratio** (shorter reformulations often mean hedging was compressed out)
- **Polarity shift** (direction of epistemic change)

In [ ]:
# Hedge and booster lexicons
STRONG_HEDGES = {'may','might','could','possibly','perhaps','potentially','presumably','putative','speculate','hypothesize','hypothesized'}
MODERATE_HEDGES = {'suggest','suggests','suggested','suggesting','indicate','indicates','indicated','indicating','appear','appears','appeared','seem','seems','seemed','propose','proposes','proposed','imply','implies','implied'}
WEAK_HEDGES = {'possible','probable','probably','likely','unlikely','potential','often','sometimes','generally','typically','approximately','roughly','believed','considered','thought'}
BOOSTERS = {'clearly','obviously','certainly','definitely','undoubtedly','demonstrates','proves','confirms','establishes','shows','revealed','determined','found','concluded','established','always','never','must','will','shall'}
ALL_HEDGES = STRONG_HEDGES | MODERATE_HEDGES | WEAK_HEDGES

def get_words(text):
    if not text: return set()
    return set(re.findall(r'\b[a-z]+\b', text.lower()))

def count_hedges(text, hedge_set):
    return len(get_words(text) & hedge_set)

def extract_features(source, reformulation, gt_tag):
    src_words = get_words(source)
    ref_words = get_words(reformulation)
    src_s = count_hedges(source, STRONG_HEDGES)
    src_m = count_hedges(source, MODERATE_HEDGES)
    src_w = count_hedges(source, WEAK_HEDGES)
    src_t = src_s + src_m + src_w
    ref_s = count_hedges(reformulation, STRONG_HEDGES)
    ref_m = count_hedges(reformulation, MODERATE_HEDGES)
    ref_w = count_hedges(reformulation, WEAK_HEDGES)
    ref_t = ref_s + ref_m + ref_w
    src_b = count_hedges(source, BOOSTERS)
    ref_b = count_hedges(reformulation, BOOSTERS)
    src_h = src_words & ALL_HEDGES
    ref_h = ref_words & ALL_HEDGES
    overlap = len(src_words & ref_words) / len(src_words | ref_words) if (src_words | ref_words) else 0
    src_len = len(source.split())
    ref_len = len(reformulation.split()) if reformulation else 0
    hedge_delta = ref_t - src_t
    if gt_tag == 'UNCERTAIN':
        polarity = -hedge_delta
    else:
        polarity = hedge_delta
    return {
        'src_strong_hedges': src_s, 'src_moderate_hedges': src_m, 'src_weak_hedges': src_w, 'src_total_hedges': src_t,
        'ref_strong_hedges': ref_s, 'ref_moderate_hedges': ref_m, 'ref_weak_hedges': ref_w, 'ref_total_hedges': ref_t,
        'hedge_preservation_rate': ref_t / src_t if src_t > 0 else 1.0,
        'hedge_delta': hedge_delta,
        'hedge_words_preserved': len(src_h & ref_h),
        'hedge_words_dropped': len(src_h - ref_h),
        'hedge_words_added': len(ref_h - src_h),
        'src_boosters': src_b, 'ref_boosters': ref_b, 'booster_delta': ref_b - src_b,
        'lexical_overlap': overlap,
        'length_ratio': ref_len / src_len if src_len > 0 else 1.0,
        'polarity_shift': polarity,
        'ground_truth_uncertain': 1 if gt_tag == 'UNCERTAIN' else 0,
    }

print('Feature extraction ready. 20 features per pair.')

In [ ]:
# Extract features from all Dimension 1 results
dataset = []
for r in dim1:
    source = r.get('text', r.get('sentence_text', ''))
    pj = r['parsed_json']
    reformulation = pj.get('reformulation', '')
    cert = pj.get('certainty_level', '').lower().strip()
    gt = r['ground_truth_tag']
    if not reformulation or not source or cert not in ('high','medium','low'):
        continue
    features = extract_features(source, reformulation, gt)
    # Label: flattened if source is UNCERTAIN but model says high certainty
    is_flat = 1 if (gt == 'UNCERTAIN' and cert == 'high') else 0
    if gt == 'UNCERTAIN' and features['hedge_words_dropped'] > 0 and features['ref_total_hedges'] == 0:
        is_flat = 1
    dataset.append({**features, 'is_flattened': is_flat, 'cert_level': cert, 'ground_truth': gt,
                    'model': r['model'], 'condition': r['condition'], 'stimulus_id': r['stimulus_id'],
                    'source_text': source, 'reformulation': reformulation})

flat_count = sum(d['is_flattened'] for d in dataset)
print(f'Dataset: {len(dataset)} pairs')
print(f'Flattened: {flat_count} ({flat_count/len(dataset)*100:.1f}%)')
print(f'Preserved: {len(dataset)-flat_count} ({(len(dataset)-flat_count)/len(dataset)*100:.1f}%)')

## Step 2: Train Classifiers

We train 3 models from scratch (no sklearn needed):
- **Logistic Regression** - simple linear boundary
- **Decision Tree** - interpretable rule-based
- **Random Forest** - ensemble of 20 trees

Train/test split is stratified by stimulus ID to prevent data leakage (same sentence never appears in both train and test).

In [ ]:
# Build feature matrix
feat_names = ['src_strong_hedges','src_moderate_hedges','src_weak_hedges','src_total_hedges',
    'ref_strong_hedges','ref_moderate_hedges','ref_weak_hedges','ref_total_hedges',
    'hedge_preservation_rate','hedge_delta','hedge_words_preserved','hedge_words_dropped',
    'hedge_words_added','src_boosters','ref_boosters','booster_delta',
    'lexical_overlap','length_ratio','polarity_shift','ground_truth_uncertain']

X = np.array([[d[f] for f in feat_names] for d in dataset])
y = np.array([d['is_flattened'] for d in dataset])

# Split by stimulus (no leakage)
stim_groups = defaultdict(list)
for i, d in enumerate(dataset):
    stim_groups[d['stimulus_id']].append(i)
stim_ids = list(stim_groups.keys())
np.random.seed(42)
np.random.shuffle(stim_ids)
split = int(len(stim_ids) * 0.8)
train_idx = [i for sid in stim_ids[:split] for i in stim_groups[sid]]
test_idx = [i for sid in stim_ids[split:] for i in stim_groups[sid]]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f'Train: {len(train_idx)} | Test: {len(test_idx)}')
print(f'Train flattened: {sum(y_train)} | Test flattened: {sum(y_test)}')

In [ ]:
# Logistic Regression (from scratch)
class LogisticRegression:
    def __init__(self, lr=0.01, epochs=1000, reg=0.1):
        self.lr, self.epochs, self.reg = lr, epochs, reg
    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))
    def fit(self, X, y):
        n, m = X.shape
        self.mean, self.std = X.mean(0), X.std(0) + 1e-8
        Xn = (X - self.mean) / self.std
        self.w, self.b = np.zeros(m), 0.0
        for _ in range(self.epochs):
            p = self.sigmoid(Xn @ self.w + self.b)
            self.w -= self.lr * ((Xn.T @ (p - y)) / n + self.reg * self.w)
            self.b -= self.lr * np.mean(p - y)
    def predict_proba(self, X):
        return self.sigmoid(((X - self.mean) / self.std) @ self.w + self.b)
    def predict(self, X, t=0.5):
        return (self.predict_proba(X) >= t).astype(int)

# Decision Tree (from scratch)
class SimpleDecisionTree:
    def __init__(self, max_depth=5): self.max_depth = max_depth
    def gini(self, y):
        if len(y) == 0: return 0
        p = np.mean(y)
        return 2 * p * (1 - p)
    def best_split(self, X, y):
        best_g, best_f, best_t = -1, None, None
        pg = self.gini(y)
        n = len(y)
        for f in range(X.shape[1]):
            for t in np.unique(X[:, f]):
                l, r = y[X[:,f]<=t], y[X[:,f]>t]
                if len(l)==0 or len(r)==0: continue
                g = pg - (len(l)/n*self.gini(l) + len(r)/n*self.gini(r))
                if g > best_g: best_g, best_f, best_t = g, f, t
        return best_f, best_t, best_g
    def build(self, X, y, d=0):
        if d >= self.max_depth or len(np.unique(y))==1 or len(y)<5:
            return {'leaf':True, 'pred':np.mean(y)}
        f, t, g = self.best_split(X, y)
        if f is None or g < 0.001: return {'leaf':True, 'pred':np.mean(y)}
        m = X[:,f] <= t
        return {'leaf':False,'feat':f,'thresh':t,'left':self.build(X[m],y[m],d+1),'right':self.build(X[~m],y[~m],d+1)}
    def fit(self, X, y): self.tree = self.build(X, y)
    def _pred(self, x, n):
        if n['leaf']: return n['pred']
        return self._pred(x, n['left'] if x[n['feat']]<=n['thresh'] else n['right'])
    def predict_proba(self, X): return np.array([self._pred(x, self.tree) for x in X])
    def predict(self, X, t=0.5): return (self.predict_proba(X)>=t).astype(int)

# Random Forest (from scratch)
class SimpleRandomForest:
    def __init__(self, n_trees=20, max_depth=5): self.n_trees, self.max_depth = n_trees, max_depth
    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            idx = np.random.choice(len(y), len(y), replace=True)
            t = SimpleDecisionTree(self.max_depth)
            t.fit(X[idx], y[idx])
            self.trees.append(t)
    def predict_proba(self, X): return np.mean([t.predict_proba(X) for t in self.trees], axis=0)
    def predict(self, X, t=0.5): return (self.predict_proba(X)>=t).astype(int)

print('Training 3 models...')
lr = LogisticRegression(lr=0.1, epochs=2000, reg=0.01)
lr.fit(X_train, y_train)
dt = SimpleDecisionTree(max_depth=5)
dt.fit(X_train, y_train)
rf = SimpleRandomForest(n_trees=20, max_depth=5)
rf.fit(X_train, y_train)
print('Done.')

## Step 3: Evaluation

We evaluate each model on the held-out test set using:
- **Accuracy** - overall correctness
- **Precision** - of the cases we flagged as flattened, how many actually were?
- **Recall** - of all actual flattening cases, how many did we catch?
- **F1** - harmonic mean of precision and recall
- **AUC-ROC** - how well does the score separate flattened from preserved?

In [ ]:
def evaluate(name, y_true, y_pred, y_proba=None):
    tp = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==1)
    fp = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==1)
    fn = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==0)
    tn = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==0)
    prec = tp/(tp+fp) if tp+fp else 0
    rec = tp/(tp+fn) if tp+fn else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0
    acc = (tp+tn)/len(y_true)
    auc = 0.0
    if y_proba is not None:
        pos, neg = y_proba[y_true==1], y_proba[y_true==0]
        if len(pos)>0 and len(neg)>0:
            for p in pos: auc += sum(1 for n in neg if p>n) + 0.5*sum(1 for n in neg if p==n)
            auc /= len(pos)*len(neg)
    print(f'\n  {name}:')
    print(f'    Accuracy:  {acc:.3f}')
    print(f'    Precision: {prec:.3f}')
    print(f'    Recall:    {rec:.3f}')
    print(f'    F1 Score:  {f1:.3f}')
    print(f'    AUC-ROC:   {auc:.3f}')
    print(f'    TP={tp} FP={fp} FN={fn} TN={tn}')
    return {'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,'auc':auc}

results_eval = {}
for name, model in [('Logistic Regression',lr), ('Decision Tree',dt), ('Random Forest',rf)]:
    results_eval[name] = evaluate(name, y_test, model.predict(X_test), model.predict_proba(X_test))

## Step 4: Feature Importance

Which features matter most for detecting flattening? The logistic regression weights tell us directly.

In [ ]:
print('Feature importance (logistic regression |weight|):\n')
coefs = sorted(zip(feat_names, np.abs(lr.w)), key=lambda x: -x[1])
for feat, w in coefs:
    bar = '#' * int(w * 30)
    print(f'  {feat:<30} {w:.4f} {bar}')

## Step 5: Demo - Try the Detector

The `detect_flattening()` function takes any (source, reformulation) pair and returns a flattening score.

**How to interpret the results:**
- Score 0.0-0.3: Epistemic stance preserved (hedging intact)
- Score 0.3-0.5: Borderline (some hedging may have shifted)
- Score 0.5-1.0: Flattened (hedging likely removed or weakened)
- `hedge_words_dropped`: how many specific hedge words disappeared
- `hedge_preservation_rate`: 1.0 = all hedges kept, 0.0 = all hedges lost

In [ ]:
def detect_flattening(source, reformulation, ground_truth_tag='UNCERTAIN'):
    features = extract_features(source, reformulation, ground_truth_tag)
    x = np.array([[features[f] for f in feat_names]])
    score = rf.predict_proba(x)[0]
    return {
        'flattening_score': round(float(score), 3),
        'classification': 'FLATTENED' if score >= 0.5 else 'PRESERVED',
        'hedge_words_dropped': features['hedge_words_dropped'],
        'hedge_words_preserved': features['hedge_words_preserved'],
        'hedge_preservation_rate': round(features['hedge_preservation_rate'], 3),
        'booster_delta': features['booster_delta'],
    }

# Demo cases
demos = [
    ('These results suggest that the protein may play a role in immune regulation.',
     'These results show that the protein plays a role in immune regulation.',
     'UNCERTAIN', 'Should detect: suggest/may dropped, show added'),
    ('These results suggest that the protein may play a role in immune regulation.',
     'These findings suggest that the protein might be involved in immune regulation.',
     'UNCERTAIN', 'Should preserve: suggest kept, might replaces may'),
    ('The experiment demonstrated that cells undergo apoptosis under these conditions.',
     'The experiment demonstrated that cells undergo apoptosis under these conditions.',
     'ASSERTIVE', 'Should preserve: no change needed'),
    ('Analysis of mRNA suggested that Tat exerts its effect on IL-2 primarily at the transcriptional level.',
     'Tat affects IL-2 at the transcriptional level.',
     'UNCERTAIN', 'Should detect: suggested dropped, sentence compressed'),
]

print('Running detector on demo cases:\n')
for src, ref, tag, note in demos:
    r = detect_flattening(src, ref, tag)
    print(f'  [{r["classification"]}] Score: {r["flattening_score"]}')
    print(f'  Source: {src[:80]}...')
    print(f'  Reform: {ref[:80]}...')
    print(f'  Hedges dropped: {r["hedge_words_dropped"]} | Preserved: {r["hedge_words_preserved"]} | Rate: {r["hedge_preservation_rate"]}')
    print(f'  Note: {note}\n')

## Step 6: Try Your Own Sentences
Paste any source sentence and its LLM reformulation to test the detector.

In [ ]:
# Edit these and re-run the cell
source = "The data suggest that this pathway may be involved in tumor suppression."
reformulation = "The data show that this pathway is involved in tumor suppression."
tag = "UNCERTAIN"  # UNCERTAIN or ASSERTIVE

result = detect_flattening(source, reformulation, tag)
print(f'Classification: {result["classification"]}')
print(f'Flattening score: {result["flattening_score"]}')
print(f'Hedge words dropped: {result["hedge_words_dropped"]}')
print(f'Hedge words preserved: {result["hedge_words_preserved"]}')
print(f'Preservation rate: {result["hedge_preservation_rate"]}')

In [ ]:
# Save everything
with open('flattening_detector_eval.json', 'w') as f:
    json.dump({'models': results_eval, 'dataset_size': len(dataset),
               'train_size': len(train_idx), 'test_size': len(test_idx),
               'flattening_rate': flat_count/len(dataset)}, f, indent=2)
print('Saved: flattening_detector_eval.json')
files.download('flattening_detector_eval.json')